In [34]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [35]:
books = pd.read_csv('Books.csv', sep=';', encoding='latin-1', on_bad_lines='skip')
users = pd.read_csv('Users.csv', sep=';', encoding='latin-1', on_bad_lines='skip')
ratings = pd.read_csv('Ratings.csv', sep=';', encoding='latin-1', on_bad_lines='skip')

/tmp/ipykernel_8407/178539441.py:2: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  users = pd.read_csv('Users.csv', sep=';', encoding='latin-1', on_bad_lines='skip')


In [36]:
books.columns

Index(['ISBN', 'Title', 'Author', 'Year', 'Publisher'], dtype='object')

In [37]:
books.head()

,ISBN,Title,Author,Year,Publisher
0,0195153448,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press
1,0002005018,Clara Callan,Richard Bruce Wright,2001,HarperFlamingo Canada
2,0060973129,Decision in Normandy,Carlo D'Este,1991,HarperPerennial
3,0374157065,Flu: The Story of the Great Influenza Pandemic...,Gina Bari Kolata,1999,Farrar Straus Giroux
4,0393045218,The Mummies of Urumchi,E. J. W. Barber,1999,W. W. Norton & Company


In [38]:
users.head()

,User-ID,Age
0,1,NaN
1,2,18
2,3,NaN
3,4,17
4,5,NaN


In [39]:
ratings.head()

,User-ID,ISBN,Rating
0,276725,034545104X,0
1,276726,0155061224,5
2,276727,0446520802,0
3,276729,052165615X,3
4,276729,0521795028,6


In [40]:
print(books.shape)
print(ratings.shape)
print(users.shape)

(271379, 5)
(1149780, 3)
(278859, 2)


In [41]:
books.isnull().sum()

,0
ISBN,0
Title,0
Author,2
Year,0
Publisher,2


In [42]:
users.isnull().sum()

,0
User-ID,0
Age,110232


In [43]:
ratings.isnull().sum()

,0
User-ID,0
ISBN,0
Rating,0


In [44]:
books.duplicated().sum()

np.int64(1)

In [45]:
books.drop_duplicates(inplace=True)

In [46]:
books.duplicated().sum()

np.int64(0)

In [47]:
ratings.duplicated().sum()

np.int64(0)

In [48]:
users.duplicated().sum()

np.int64(0)

In [49]:
x = ratings['User-ID'].value_counts() > 200

In [50]:
x[x].shape #these unique users have given ratings for 200+ books

(899,)

In [51]:
y = x[x].index

In [52]:
y

Index([ 11676, 198711, 153662,  98391,  35859, 212898, 278418,  76352, 110973,
       235105,
       ...
       116122,  44296,  28634,  59727,  73681, 274808, 188951,   9856, 155916,
       268622],
      dtype='int64', name='User-ID', length=899)

In [53]:
print(type(ratings))

<class 'pandas.core.frame.DataFrame'>


In [54]:
ratings = ratings[ratings['User-ID'].isin(y)]

In [58]:
ratings.shape

(526356, 3)

# Popularity Based Recommender System

In [59]:
ratings_with_books = ratings.merge(books,on='ISBN')

In [60]:
ratings_with_books.sample(5)

,User-ID,ISBN,Rating,Title,Author,Year,Publisher
227508,129851,051757439X,0,Hats: A Stylish History and Collector's Guide,Jody Shields,1991,Clarkson N Potter Publishers
458108,257028,013920430X,7,"Three Genres: The Writing of Poetry, Fiction, ...",Stephen Minot,1987,Prentice Hall
391117,226545,1551667304,0,Undressed,Stef Ann Holm,2003,Mira
458875,257204,0451192303,0,Between a Wok and a Hard Place: A Pennsylvania...,Tamar Myers,1998,Signet Book
187987,108005,0345445848,0,Big Cherry Holler: A Big Stone Gap Novel (Ball...,Adriana Trigiani,2002,Ballantine Books


In [62]:
ratings_with_books.shape #books with no info were deleted while merging

(487685, 7)

In [72]:
number_rating = ratings_with_books.groupby('Title')['Rating'].count().reset_index()

In [73]:
print(number_rating)

                                                    Title  Rating
0        A Light in the Storm: The Civil War Diary of ...       2
1                                   Always Have Popsicles       1
2                    Apple Magic (The Collector's series)       1
3        Beyond IBM: Leadership Marketing and Finance ...       1
4        Clifford Visita El Hospital (Clifford El Gran...       1
...                                                   ...     ...
160275  Ã?Ã?ber die Pflicht zum Ungehorsam gegen den...       3
160276                                    Ã?Ã?lpiraten.       1
160277                   Ã?Ã?rger mit Produkt X. Roman.       1
160278                            Ã?Ã?stlich der Berge.       1
160279                                Ã?Ã?thique en toc       1

[160280 rows x 2 columns]


In [74]:
number_rating.rename(columns={'Rating':'number of ratings'}, inplace=True)

In [77]:
final_rating = ratings_with_books.merge(number_rating,on='Title')

In [79]:
final_rating.shape

(487685, 8)

In [81]:
final_rating = final_rating[final_rating['number of ratings'] >= 50]

In [82]:
final_rating

,User-ID,ISBN,Rating,Title,Author,Year,Publisher,number of ratings
0,277427,002542730X,10,Politically Correct Bedtime Stories: Modern Ta...,James Finn Garner,1994,John Wiley & Sons Inc,82
13,277427,0060930535,0,The Poisonwood Bible: A Novel,Barbara Kingsolver,1999,Perennial,133
15,277427,0060934417,0,Bel Canto: A Novel,Ann Patchett,2002,Perennial,108
18,277427,0061009059,9,One for the Money (Stephanie Plum Novels (Pape...,Janet Evanovich,1995,HarperTorch,108
24,277427,006440188X,0,The Secret Garden,Frances Hodgson Burnett,1998,HarperTrophy,79
...,...,...,...,...,...,...,...,...
487519,275970,1400031354,0,Tears of the Giraffe (No.1 Ladies Detective Ag...,Alexander McCall Smith,2002,Anchor,84
487520,275970,1400031362,0,Morality for Beautiful Girls (No.1 Ladies Dete...,Alexander McCall Smith,2002,Anchor,60
487593,275970,1573229725,0,Fingersmith,Sarah Waters,2002,Riverhead Books,59
487632,275970,1586210661,9,Me Talk Pretty One Day,David Sedaris,2001,Time Warner Audio Major,146


In [83]:
final_rating.drop_duplicates(['User-ID','Title'],inplace=True)

/tmp/ipykernel_8407/151764437.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_rating.drop_duplicates(['User-ID','Title'],inplace=True)


In [84]:
final_rating.shape # the dataset where every book has more than 50 ratings

(59850, 8)

In [85]:
book_pivot = final_rating.pivot_table(columns='User-ID',index='Title',values='Rating')

In [86]:
book_pivot

User-ID,254,2276,2766,2977,3363,3757,4017,4385,6242,6251,...,274004,274061,274301,274308,274808,275970,277427,277478,277639,278418
Title,,,,,,,,,,,,,,,,,,,,,
1984,9.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN
1st to Die: A Novel,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2nd Chance,NaN,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0,NaN
4 Blondes,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
84 Charing Cross Road,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,10.0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Year of Wonders,NaN,NaN,NaN,7.0,NaN,NaN,NaN,NaN,7.0,NaN,...,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN
You Belong To Me,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Zen and the Art of Motorcycle Maintenance: An Inquiry into Values,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0,...,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN


In [90]:
book_pivot.fillna(0, inplace=True)

In [91]:
book_pivot

User-ID,254,2276,2766,2977,3363,3757,4017,4385,6242,6251,...,274004,274061,274301,274308,274808,275970,277427,277478,277639,278418
Title,,,,,,,,,,,,,,,,,,,,,
1984,9.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1st to Die: A Novel,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2nd Chance,0.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4 Blondes,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
84 Charing Cross Road,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,10.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Year of Wonders,0.0,0.0,0.0,7.0,0.0,0.0,0.0,0.0,7.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
You Belong To Me,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Zen and the Art of Motorcycle Maintenance: An Inquiry into Values,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [93]:
from scipy.sparse import csr_matrix
book_sparse = csr_matrix(book_pivot)

In [95]:
type(book_sparse)

scipy.sparse._csr.csr_matrix

In [96]:
from sklearn.neighbors import NearestNeighbors
model = NearestNeighbors(algorithm='brute')

In [97]:
model.fit(book_sparse)

NearestNeighbors(algorithm='brute')

In [99]:
distances, suggestions = model.kneighbors(book_pivot.iloc[237, :].values.reshape(1, -1), n_neighbors = 6)

In [101]:
suggestions

array([[237, 238, 240, 241, 184, 536]])

In [106]:
book_pivot.index[237]

'Harry Potter and the Chamber of Secrets (Book 2)'

In [104]:
for i in range(len(suggestions)):
  print(book_pivot.index[suggestions[i]])

Index(['Harry Potter and the Chamber of Secrets (Book 2)',
       'Harry Potter and the Goblet of Fire (Book 4)',
       'Harry Potter and the Prisoner of Azkaban (Book 3)',
       'Harry Potter and the Sorcerer's Stone (Book 1)', 'Exclusive',
       'The Cradle Will Fall'],
      dtype='object', name='Title')


In [113]:
def recommend(n):
  print(book_pivot.index[n])
  distances, suggestions = model.kneighbors(book_pivot.iloc[n, :].values.reshape(1, -1), n_neighbors = 6)

  for i in range(len(suggestions)):
    return (book_pivot.index[suggestions[i]])


In [112]:
recommend(54)

Animal Farm


('\n',
 Index(['Animal Farm', 'Exclusive', 'Jacob Have I Loved', 'Second Nature',
        'Pleading Guilty', 'No Safe Place'],
       dtype='object', name='Title'))

In [117]:
np.where(book_pivot.index == 'Animal Farm')[0][0]

np.int64(54)

In [130]:
def recommend_book(book_name):
  book_id = np.where(book_pivot.index == book_name)[0][0]
  print(f"The book_id is: {book_id}")
  # print(book_pivot.index[book_id])
  distances, suggestions = model.kneighbors(book_pivot.iloc[book_id, :].values.reshape(1, -1), n_neighbors = 6)

  print(f"The Suggestions for {book_name} are: ")
  for i in range(len(suggestions)):
    print(book_pivot.index[suggestions[i]])


In [131]:
recommend_book('Animal Farm')

The book_id is: 54
The Suggestions for Animal Farm are: 
Index(['Animal Farm', 'Exclusive', 'Jacob Have I Loved', 'Second Nature',
       'Pleading Guilty', 'No Safe Place'],
      dtype='object', name='Title')
